In [1]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma pillow open_clip_torch torch matplotlib unstructured pydantic
import os
from textbook_loading import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
    delete_irrelevant_images,
)

In [2]:
pdf_file = './data/Cat_Owners_Home_Veterinary_Handbook_Trimed.pdf'
image_output_dir = './figures/Cat_Owners_Home_Veterinary_Handbook'
chroma_persist_dir = './chroma/Cat_Owners_Home_Veterinary_Handbook/'

# Make sure the data directory exists
assert os.path.exists('./data'), "Error: './data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [3]:
print("📝 Unstructuring textbooks, filtering junks, semanic chunking...")
raw_pdf_elements = load_book(pdf_file, image_output_dir)
print("🎉 1.process_pdf_with_semantic_chunking complete.")


📝 Unstructuring textbooks, filtering junks, semanic chunking...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


🎉 1.process_pdf_with_semantic_chunking complete.


In [4]:
# Clean and categorize
texts, tables, images_raw = clean_and_categorize_elements(raw_pdf_elements, window_size=2, min_meaningful_text_length=75)

In [5]:
# Summarize, store, etc.
text_summaries, table_summaries, image_paths, relevant_images_to_summarize, image_summaries = summarize_elements(
    texts, tables, images_raw
)

/Users/mas/Desktop/LLM_Veterinary_AI/textbook_loading.py:257: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")


Texts and Tables Summary Done!
Checking image relevance with local textual context...
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-1-1.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-3-6.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-5-10.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-8-14.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-9-18.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-10-19.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-12-22.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-13-24.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-14-25.jpg
Skipping decorative image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-15-26.jpg
Skipping decorative i

In [6]:
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables, image_paths,
    relevant_images_to_summarize, image_summaries,
    persist_directory=chroma_persist_dir
)

In [7]:
delete_irrelevant_images(images_raw, relevant_images_to_summarize)

Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-1-1.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-3-6.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-5-10.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-8-14.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-9-18.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-10-19.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-12-22.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-13-24.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figure-14-25.jpg
Successfully deleted irrelevant image: ./figures/Cat_Owners_Home_Veterinary_Handbook/figu

In [8]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

0

# Inspecting Retrieved Docs

In [9]:
query = "My cat has being scratching its ear too often. There are some dark greasy thing in it. It sratch its ear so often and so hard that I see wounds and blood in it. What should I do?"
results = retriever.retrieve_multi_modal(query, k=5)

In [10]:
from IPython.display import display, HTML
import os

# 1. Display all images together as thumbnails
image_paths = set()
for res in results:
    if res["modality"] == "image" and os.path.exists(res["summary"]):
        image_paths.add(res["summary"])
    elif res["modality"] == "image_summary":
        img_path = res["original_metadata"].get("image_path")
        if img_path and os.path.exists(img_path):
            image_paths.add(img_path)

if image_paths:
    html_imgs = " ".join(
        f'<img src="{img}" width="100" style="margin:2px; border:1px solid #ccc;">' for img in image_paths
    )
    display(HTML(html_imgs))
else:
    print("No images found in results.")

# 2. Display original text for each text result
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "docstore"):
            doc = retriever.docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)

---------------------------------------- Retrieved Text Chunks (first 300 chars) ----------------------------------------
**Ear Mites:**

* Symptoms: intense itching, scratching, head shaking, dark brown waxy discharge
* Treatment: medication chosen by veterinarian (e.g., Nolvamite, Mitaclear), complete recommended course of treatment
* Prevention: clean ears, treat entire cat with topical insecticide preparation

**Ba...
--------------------
**Ear Care Summary**

* Clean ears when there's excessive wax, dirt, or debris.
* For small amounts of wax, use a damp cotton ball or swab.
* For dirty ears, use an ear-cleaning solution (e.g., Oti-Clens) or mineral oil/olive oil/vinegar mixture.
* Massage the base of the ear to loosen debris and wi...
--------------------
Here's the summary:

**Nerve Damage**

* Rarely occurs after removal
* Most cats have complete recovery

**Ceruminous Gland Problems**

* Benign cysts are common in cat ears, but can block ear canal if not removed
* Malignant tu